# Mechanistic figures (publication)

- **Figure M1 -- activation patching.** Columns: tulu drop, alpaca drop, tulu recovery. Top: restoration of the
  first-token logit lean, median over flipping capitals with the interquartile band. Bottom: share of flipping
  capitals whose *generated* answer is the context answer. Layers 19-26 (where the logit lens first reads the
  answer) are shaded.
- **Figure M2 -- logit lens.** Mean first-token lean per layer for flipping vs stable capitals at each phase.

Inputs (local): `activation-patching/`, `logit-lens-v2/`. Output: `figures/`.

In [ ]:
import json, glob, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Runs locally from src/mechanistic-analysis/. In Colab, use the repo clone (results are committed to GitHub).
if os.path.isdir("/content"):
    REPO = "/content/context-parametric-inversion-research"
    if not os.path.isdir(REPO):
        os.system(f"git clone --branch dev https://github.com/GIRIAYUSH/context-parametric-inversion-research.git {REPO}")
    os.system(f"git -C {REPO} pull -q origin dev")
    os.chdir(f"{REPO}/src/mechanistic-analysis")
assert os.path.isdir("activation-patching") and os.path.isdir("logit-lens-v2"), \
    f"result folders not found in {os.getcwd()}"

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

# validated palette (dataviz skill, references/palette.md) + chart ink
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK_PRIMARY, INK_SECONDARY, INK_MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE, SURFACE, SHADE = "#e1e0d9", "#c3c2b7", "#fcfcfb", "#efe9d2"

plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial"],
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK_SECONDARY,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED, "text.color": INK_PRIMARY,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.grid": True, "axes.axisbelow": True,
})

def style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=0)

def save(fig, name):
    fig.savefig(f"{FIG_DIR}/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{FIG_DIR}/{name}.pdf", bbox_inches="tight")

COMPARISONS = [("tulu", "peak", "trough", "Tulu: peak → trough"),
               ("alpaca", "peak", "trough", "Alpaca: peak → trough"),
               ("tulu", "recovery", "trough", "Tulu: recovery → trough")]

## Figure M1 -- activation patching

In [ ]:
def load(run, src, tgt, suffix=""):
    return json.load(open(f"activation-patching/{run}_seed0_{src}_to_{tgt}{suffix}.json", encoding="utf-8"))

fig, axes = plt.subplots(2, 3, figsize=(12, 6.2), sharex=True, sharey="row",
                         gridspec_kw={"height_ratios": [1.15, 1]})

for col, (run, src, tgt, title) in enumerate(COMPARISONS):
    # top: logit restoration
    d = load(run, src, tgt)
    L = np.array(d["layers"])
    R = np.array([(np.array(v["D_patched"]) - v["D_target"]) / (v["D_source"] - v["D_target"]) for v in d["items"].values()])
    med = np.median(R, axis=0)
    q25, q75 = np.percentile(R, [25, 75], axis=0)
    ax = axes[0, col]
    ax.axvspan(19, 26, color=SHADE, zorder=0)
    ax.axhline(0, color=BASELINE, linewidth=1)
    ax.axhline(1, color=INK_MUTED, linewidth=0.8, linestyle=":")
    ax.fill_between(L, q25, q75, color=BLUE, alpha=0.18, linewidth=0, zorder=2)
    ax.plot(L, med, color=BLUE, linewidth=2.2, zorder=3)
    ax.set_title(f"{title}   (n = {len(R)})", fontsize=11.5, loc="left", color=INK_PRIMARY, pad=10)
    style(ax)

    # bottom: generated answer
    g = load(run, src, tgt, "_generation")
    rows = list(g["items"].values())
    share = lambda cond: 100 * np.mean([r[cond]["label"] == "CTX" for r in rows])
    GL = g["layers"]
    ax = axes[1, col]
    ax.axvspan(19, 26, color=SHADE, zorder=0)
    ax.axhline(share("source"), color=AQUA, linewidth=1.2, linestyle="--", zorder=2, label="unpatched source model")
    ax.axhline(share("target"), color=ORANGE, linewidth=1.2, linestyle="--", zorder=2, label="unpatched trough model")
    ax.plot(GL, [share(f"layer{l}") for l in GL], color=BLUE, linewidth=2.2, marker="o", markersize=5, zorder=3,
            label="trough, patched at layer")
    style(ax)
    ax.set_xlabel("Patched layer", fontsize=10)

axes[1, 0].legend(frameon=False, fontsize=8.5, loc="center right")

axes[0, 0].set_ylabel("Restoration of first-token lean\n(0 = trough, 1 = source)", fontsize=10)
axes[1, 0].set_ylabel("Generates the context\nanswer (% of items)", fontsize=10)
axes[1, 0].yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
axes[1, 0].set_ylim(-5, 108)
axes[0, 0].text(22.5, axes[0, 0].get_ylim()[1] * 0.97, "answer first\nreadable\n(logit lens)", fontsize=8,
                color=INK_SECONDARY, ha="center", va="top")
for ax in axes[1]:
    ax.set_xticks([1, 4, 8, 12, 16, 20, 24, 28, 31])

fig.suptitle("Patching the source model's residual stream into the trough model restores context-following",
             fontsize=13, x=0.06, ha="left", y=1.02, color=INK_PRIMARY)
fig.tight_layout()
save(fig, "patching")
plt.show()

## Figure M2 -- logit lens

In [ ]:
PHASES = {"tulu": ["peak", "trough", "recovery"], "alpaca": ["peak", "trough"]}
PHASE_COLOR = {"peak": BLUE, "trough": ORANGE, "recovery": AQUA}

RAW = {}
for f in glob.glob("logit-lens-v2/*.json"):
    d = json.load(open(f, encoding="utf-8"))
    RAW[(d["run"], d["phase"])] = {i: np.array(v["D"]) for i, v in d["curves"].items() if not v["same_first_token"]}

def groups(run):
    phases = PHASES[run]
    ids = [i for i in RAW[(run, "peak")] if i.startswith("cap_") and all(i in RAW[(run, p)] for p in phases)]
    fin = {p: {i: RAW[(run, p)][i][-1] for i in ids} for p in phases}
    return {"Flipping capitals": [i for i in ids if fin["peak"][i] > 0 and fin["trough"][i] < 0],
            "Stable capitals":   [i for i in ids if all(fin[p][i] > 0 for p in phases)]}

fig, axes = plt.subplots(2, 2, figsize=(11, 6.6), sharex=True, sharey=True)
for row, run in enumerate(["tulu", "alpaca"]):
    for col, (gname, ids) in enumerate(groups(run).items()):
        ax = axes[row, col]
        ax.axvspan(19, 26, color=SHADE, zorder=0)
        ax.axhline(0, color=BASELINE, linewidth=1)
        for p in PHASES[run]:
            ax.plot(np.mean([RAW[(run, p)][i] for i in ids], axis=0), color=PHASE_COLOR[p], linewidth=2.2, label=p)
        ax.set_title(f"{run.capitalize()} — {gname.lower()} (n = {len(ids)})", fontsize=11.5, loc="left", pad=8)
        style(ax)
        if row == 1:
            ax.set_xlabel("Layer", fontsize=10)
        if col == 0:
            ax.set_ylabel("Mean lean (logits)\n+ context answer  /  − own answer", fontsize=10)
axes[0, 0].legend(frameon=False, fontsize=9, loc="upper left")
fig.suptitle("Logit lens: the answer becomes readable from layer ~19; at the trough, flipping items lean to their own answer",
             fontsize=13, x=0.06, ha="left", y=1.02)
fig.tight_layout()
save(fig, "logit_lens")
plt.show()